# Spectrogram Generation

In this notebook, the previously labeled vibration time-series segments are transformed into time–frequency representations using short-time Fourier transform (STFT). The resulting spectrograms are normalized and converted into RGB images by stacking the X, Y, and Z acceleration axes. These images serve as input for the subsequent convolutional neural network (CNN) used for chatter detection.

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import spectrogram
from PIL import Image
import os
from tqdm import tqdm
import random

# In-/ Output Structure & Class-Mapping

In [7]:
DATA_ROOT = Path("../data/01_windowed_labeled_2,5s")
OUTPUT_ROOT = Path("../data/03_spectrograms_150x100px_dataset")

CLASS_MAP = {
    "chatter": "chatter",
    "no_chatter": "no_chatter" 
}

SEED = 42 #seed for splitting train and validation data
TRAIN_FRAC = 0.8 # percentage of normal spectrograms used for training 

# Spectrogram configuration
- NPERSEG: controls frequency resolution of spectrogram
larger values → better frequency resolution, lower time resolution
- MAX_FREQ_CUT: limits analysis to relevant chatter frequency range
- DB_MIN / DB_MAX: spectrogram is converted to decibel scale --> fixed scaling for energy
- TARGET_IMG_SIZE: image size of three color spectrogram

In [ ]:
NPERSEG = 512#2048
MAX_FREQ_CUT = 5000
DB_MIN = -75
DB_MAX = -20
EPS = 1e-12

TARGET_IMG_SIZE = (150, 100)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## Spectrogram generator function

In [9]:
def create_spectrogram(sig, fs):
    noverlap = int(0.75 * NPERSEG)

    f, t, Sxx = spectrogram(
        sig,
        fs=fs,
        nperseg=NPERSEG,
        noverlap=noverlap,
        mode="psd"
    )

    mask = f <= MAX_FREQ_CUT
    Sxx = Sxx[mask, :]

    Sxx_db = 10 * np.log10(Sxx + EPS)
    Sxx_db = np.clip(Sxx_db, DB_MIN, DB_MAX)

    return 1.0 - (Sxx_db - DB_MIN) / (DB_MAX - DB_MIN)

In [10]:
npz_files = list(DATA_ROOT.rglob("*.npz"))
print(f"Found {len(npz_files)} npz files")

# Partitioniere nach label_folder (muss mit CLASS_MAP zusammenpassen)
no_chatter_npz = []
chatter_npz = []

for p in npz_files:
    data = np.load(p)
    label = str(data["label"])
    label_folder = CLASS_MAP.get(label, "unknown")
    data.close()

    if label_folder == "no_chatter":
        no_chatter_npz.append(p)
    elif label_folder == "chatter":
        chatter_npz.append(p)
    else:
        print(f"Skipping unexpected label_folder='{label_folder}' for file: {p}")

# Deterministisches Random Split nur für no_chatter
rng = random.Random(SEED)
rng.shuffle(no_chatter_npz)

n_train = int(len(no_chatter_npz) * TRAIN_FRAC)
train_no_chatter_npz = no_chatter_npz[:n_train]
val_no_chatter_npz = no_chatter_npz[n_train:]

print("no_chatter train:", len(train_no_chatter_npz))
print("no_chatter validation:", len(val_no_chatter_npz))
print("chatter validation:", len(chatter_npz))

Found 651 npz files
no_chatter train: 472
no_chatter validation: 118
chatter validation: 61


# RGB Image Conversion
This function converts the labeled vibration time-series segment into a 3-channel spectrogram image.
Combines three spectrograms into a single 3-channel image:

R = X-axis
G = Y-axis
B = Z-axis

Each vibration segment is transformed into a three-channel spectrogram image, where each channel corresponds to a different sensor axis. This encoding preserves spatial vibration directionality while enabling convolutional neural networks to learn cross-axis correlations relevant for chatter detection.

**Advantages of these representation:**
- harmonic stripes (chatter)
- broadband noise (instability)
- axis coupling effects

In [13]:
def process_npz(npz_path, split_folder, target_label_folder):
    data = np.load(npz_path)

    x = data["X"]
    y = data["Y"]
    z = data["Z"]

    label = str(data["label"])
    # label_folder wird nicht mehr aus label abgeleitet, sondern kommt vom Caller
    fs = 50000  # fallback

    specs = [
        create_spectrogram(x, fs),
        create_spectrogram(y, fs),
        create_spectrogram(z, fs)
    ]

    rgb = np.stack(specs, axis=-1)
    rgb = np.clip(rgb, 0, 1)
    rgb = np.flipud(rgb)

    img = Image.fromarray((rgb * 255).astype(np.uint8))
    img = img.resize(TARGET_IMG_SIZE)

    experiment = npz_path.parent.parent.name
    out_name = f"{experiment}_{npz_path.stem}_{label}.png"

    out_path = OUTPUT_ROOT / split_folder / target_label_folder / out_name
    out_path.parent.mkdir(parents=True, exist_ok=True)
    img.save(out_path)

In [14]:
# Train: 80% no_chatter
for npz_file in tqdm(train_no_chatter_npz, desc="Processing train/no_chatter"):
    try:
        process_npz(npz_file, "train", "no_chatter")
    except Exception as e:
        print(f"Error in {npz_file}: {e}")

# Validation: 20% no_chatter
for npz_file in tqdm(val_no_chatter_npz, desc="Processing validation/no_chatter"):
    try:
        process_npz(npz_file, "validation", "no_chatter")
    except Exception as e:
        print(f"Error in {npz_file}: {e}")

# Validation: all chatter
for npz_file in tqdm(chatter_npz, desc="Processing validation/chatter"):
    try:
        process_npz(npz_file, "validation", "chatter")
    except Exception as e:
        print(f"Error in {npz_file}: {e}")

Processing validation/chatter: 100%|██████████| 61/61 [00:17<00:00,  3.45it/s]
